<a href="https://colab.research.google.com/github/kiryu2253-del/q-learning-maze/blob/main/q_learning_maze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 5×5の迷路を作る
# 0 = 通れる道，1 = 壁
maze = [
    [0, 0, 1, 0, 0],
    [1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
]

# 迷路を1行ずつ表示する
for row in maze:
    print(row)


## 2. 迷路を図として表示する

数字だけでは迷路が分かりにくいので，Matplotlibを使ってマス目として表示する．

- `0`：通れる道
- `1`：壁
- `S`：スタート
- `G`：ゴール


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# スタート位置とゴール位置
start = (0, 0)
goal = (4, 4)

# listで作った迷路をNumPy配列に変換
maze_array = np.array(maze)

# 迷路を表示
fig, ax = plt.subplots()
ax.imshow(maze_array, cmap="binary")

# スタートとゴールを表示
ax.text(start[1], start[0], "S", ha="center", va="center", fontsize=18)
ax.text(goal[1], goal[0], "G", ha="center", va="center", fontsize=18)

# マス目を見やすくする
ax.set_xticks(np.arange(-0.5, 5, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 5, 1), minor=True)
ax.grid(which="minor", linewidth=1)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.show()


## 3. エージェントを迷路に置く

まずはAI学習を始める前に，迷路の中にエージェント `P` を置く．
エージェントの位置は `(行, 列)` で表す．


In [ ]:
# エージェントの現在位置
agent = start

def show_maze(agent_position):
    fig, ax = plt.subplots()
    ax.imshow(maze_array, cmap="binary")

    # スタート・ゴール・エージェントを表示
    ax.text(start[1], start[0], "S", ha="center", va="center", fontsize=18)
    ax.text(goal[1], goal[0], "G", ha="center", va="center", fontsize=18)
    ax.text(agent_position[1], agent_position[0], "P", ha="center", va="center", fontsize=18)

    # マス目を表示
    ax.set_xticks(np.arange(-0.5, 5, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 5, 1), minor=True)
    ax.grid(which="minor", linewidth=1)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    plt.show()

show_maze(agent)


## 4. エージェントを1マス動かす

Q学習では，エージェントが選べる行動を `Action` と呼ぶ．
今回は次の4つの行動を使う．

- `0`：上
- `1`：右
- `2`：下
- `3`：左

まずはAIに選ばせる前に，人間が指定した行動で1マス動かしてみる．


In [ ]:
# 行動番号と移動方向
actions = {
    0: (-1, 0),  # 上
    1: (0, 1),   # 右
    2: (1, 0),   # 下
    3: (0, -1)   # 左
}

def move(agent_position, action):
    row, col = agent_position
    dr, dc = actions[action]

    new_row = row + dr
    new_col = col + dc

    # 迷路の外に出る場合は動かない
    if new_row < 0 or new_row >= 5 or new_col < 0 or new_col >= 5:
        return agent_position

    # 壁なら動かない
    if maze[new_row][new_col] == 1:
        return agent_position

    return (new_row, new_col)

# スタート地点から右へ1マス動かす
agent = start
agent = move(agent, 1)

print("移動後の位置:", agent)
show_maze(agent)


## 5. ランダムに迷路を歩かせる

ここではまだ学習は行わず，エージェントに上下左右の行動をランダムに選ばせる．
Q学習を始める前の「何も学習していない状態」を確認する．


In [ ]:
import random

# スタート地点に戻す
agent = start

# 10回ランダムに行動する
for step in range(10):
    action = random.randint(0, 3)
    agent = move(agent, action)

    print("step", step + 1, "action", action, "position", agent)

# 最後の位置を表示
show_maze(agent)


## 6. Qテーブルを作る

Q学習では，各状態（マス）で各行動（上・右・下・左）がどれくらい良いかを `Q値` として保存する．
今回は 5×5 = 25 個の状態と，4 個の行動を使うため，Qテーブルは `25 × 4` になる．

学習前は何も知らないので，すべてのQ値を0から始める．


In [ ]:
import numpy as np

# 状態数：5×5 = 25マス
num_states = 25

# 行動数：上・右・下・左の4種類
num_actions = 4

# Qテーブルをすべて0で初期化
Q = np.zeros((num_states, num_actions))

print("Qテーブルの形:", Q.shape)
print(Q)


### 状態番号の付け方

迷路の `(行, 列)` を，Qテーブルで使う1つの状態番号に変換する．

例：
- `(0, 0)` → 状態0
- `(0, 1)` → 状態1
- `(1, 0)` → 状態5
- `(4, 4)` → 状態24


In [ ]:
def position_to_state(position):
    row, col = position
    return row * 5 + col

print("スタートの状態番号:", position_to_state(start))
print("ゴールの状態番号:", position_to_state(goal))
